In [1]:
import numpy as np
import pandas as pd
import time 
import warnings

warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv('../data/processed/df_productivity_score.csv')
df.head()

,Age,Years_Experience,WFH_Days_Per_Week,Gender,Education_Level,Marital_Status,Has_Children,Location_Type,Department,Job_Level,...,Team_Collaboration_Frequency,Productivity_Score,Innovation_Score,Efficiency_Rating,Meetings_Per_Week,Commute_Time_Minutes,Job_Satisfaction,Stress_Level,Work_Life_Balance,Smart_Work_Index
0,39,10,2,Female,Associate Degree,Married,Yes,Urban,Product,Mid-Level,...,Few times per week,52.2,52.1,72.1,4,48,55.9,6,8,2948.86
1,33,4,5,Female,Master Degree,Married,No,Urban,Customer Success,Senior,...,Monthly,81.5,77.9,89.5,12,0,96.1,3,8,5515.32
2,40,3,3,Male,PhD,Single,Yes,Rural,Operations,Mid-Level,...,Few times per week,82.2,63.2,95.0,15,24,90.4,5,6,5176.08
3,48,14,3,Male,Bachelor Degree,Married,Yes,Urban,Finance,Manager,...,Daily,75.6,82.5,95.0,8,8,100.0,10,5,5791.50
4,32,6,5,Male,High School,Divorced,Yes,Rural,Engineering,Senior,...,Few times per week,98.0,67.5,95.0,10,0,100.0,3,4,6628.50


In [4]:
X = df.drop(columns=['Productivity_Score'])
y = df['Productivity_Score']

X.shape, y.shape

((1440, 25), (1440,))

In [ ]:
cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
num_cols = X.drop(columns=cat_cols).columns.tolist()

print('Full columns', len(X.columns.tolist()))
print("Categorical columns:", len(cat_cols))
print("Numerical columns:", len(num_cols))

Full columns 25
Categorical columns: 13
Numerical columns: 12


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=X['Gender'])

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((1152, 25), (1152,), (288, 25), (288,))

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)]
)

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)   
X_train_processed.shape, X_test_processed.shape

((1152, 78), (288, 78))

In [11]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.ensemble import RandomForestRegressor,HistGradientBoostingRegressor,GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor  

from sklearn.metrics import mean_absolute_error, r2_score,root_mean_squared_error
import time

modellar ={
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'ElasticNet Regression': ElasticNet(),
    'Random Forest Regressor': RandomForestRegressor(),
    'Gradient Boosting Regressor': GradientBoostingRegressor(),
    'Hist Gradient Boosting Regressor': HistGradientBoostingRegressor(),
    'K-Nearest Neighbors Regressor': KNeighborsRegressor(),
    'Support Vector Regressor': SVR(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'MLP Regressor': MLPRegressor(max_iter=500,random_state=42),
    
    'XGBoost Regressor': XGBRegressor(objective='reg:squarederror', eval_metric='rmse'),
    'LightGBM Regressor': LGBMRegressor(objective='regression', metric='rmse',verbose=-1),
    'CatBoost Regressor': CatBoostRegressor(verbose=0, objective='RMSE')
}

In [12]:
from sklearn.metrics import mean_absolute_percentage_error

In [15]:
natijalar = []
for model_name, model in modellar.items():
    print(f"Training {model_name}...")
    start_time = time.time()
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    end_time = time.time()
    
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    
    natijalar.append({
        'Model': model_name,
        'MAE': mae,
        'R2 Score': r2,
        'RMSE': rmse,
        'MAPE': mape,
        'Training Time (s)': end_time - start_time
    })
natijalar_df = pd.DataFrame(natijalar)

Training Linear Regression...
Training Ridge Regression...
Training Lasso Regression...
Training ElasticNet Regression...
Training Random Forest Regressor...
Training Gradient Boosting Regressor...
Training Hist Gradient Boosting Regressor...
Training K-Nearest Neighbors Regressor...
Training Support Vector Regressor...
Training Decision Tree Regressor...
Training MLP Regressor...
Training XGBoost Regressor...
Training LightGBM Regressor...
Training CatBoost Regressor...


In [19]:
natijalar_df.sort_values(by='R2 Score', ascending=False)

,Model,MAE,R2 Score,RMSE,MAPE,Training Time (s)
5,Gradient Boosting Regressor,3.643541,0.892664,4.782709,0.047417,0.294268
4,Random Forest Regressor,3.753108,0.889066,4.862216,0.048515,0.926918
13,CatBoost Regressor,3.736601,0.887219,4.902523,0.048296,1.583026
6,Hist Gradient Boosting Regressor,3.762465,0.886002,4.928899,0.048622,0.259574
12,LightGBM Regressor,3.765934,0.882642,5.001011,0.048704,0.061160
10,MLP Regressor,3.966923,0.877754,5.104086,0.049572,1.522542
0,Linear Regression,4.016315,0.876446,5.131337,0.050201,0.013451
1,Ridge Regression,4.016429,0.876387,5.132553,0.050199,0.001733
11,XGBoost Regressor,4.128916,0.864040,5.382790,0.053245,0.069520
2,Lasso Regression,4.262210,0.862536,5.412484,0.054239,0.004313


In [22]:
from sklearn.model_selection import cross_validate

model = GradientBoostingRegressor(random_state=42)

cv = cross_validate(model, X_train_processed, y_train, cv=5, scoring=['r2', 'neg_root_mean_squared_error','neg_mean_absolute_percentage_error'], return_train_score=True,verbose=2)
cv

[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.1s finished


{'fit_time': array([0.22253799, 0.2185235 , 0.24049902, 0.23733974, 0.21570969]),
 'score_time': array([0.00200462, 0.00301051, 0.00200558, 0.        , 0.01907706]),
 'test_r2': array([0.90714352, 0.91055221, 0.87668848, 0.8744276 , 0.8845565 ]),
 'train_r2': array([0.9471604 , 0.94339331, 0.95080459, 0.95164184, 0.9502772 ]),
 'test_neg_root_mean_squared_error': array([-4.82747662, -4.65571453, -5.23819437, -5.04389483, -5.18038583]),
 'train_neg_root_mean_squared_error': array([-3.45183392, -3.58847635, -3.3805698 , -3.38948535, -3.38274983]),
 'test_neg_mean_absolute_percentage_error': array([-0.04907631, -0.04761331, -0.05074522, -0.05077888, -0.05184391]),
 'train_neg_mean_absolute_percentage_error': array([-0.03370004, -0.03495619, -0.03336139, -0.03290764, -0.03294048])}

In [23]:
cvdf = pd.DataFrame(cv)
cvdf

,fit_time,score_time,test_r2,train_r2,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error,test_neg_mean_absolute_percentage_error,train_neg_mean_absolute_percentage_error
0,0.222538,0.002005,0.907144,0.947160,-4.827477,-3.451834,-0.049076,-0.033700
1,0.218524,0.003011,0.910552,0.943393,-4.655715,-3.588476,-0.047613,-0.034956
2,0.240499,0.002006,0.876688,0.950805,-5.238194,-3.380570,-0.050745,-0.033361
3,0.237340,0.000000,0.874428,0.951642,-5.043895,-3.389485,-0.050779,-0.032908
4,0.215710,0.019077,0.884557,0.950277,-5.180386,-3.382750,-0.051844,-0.032940


In [35]:
tf = cvdf['test_r2'].mean()
tg = cvdf['train_r2'].mean()

round(float((tg-tf)*100), 3)

5.798

In [39]:
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed) 
r2 = r2_score(y_test, y_pred)
r2   

0.8926193652783236

In [40]:
import joblib

joblib.dump(model, '../models/Productivity/GradientBoostingRegressor.joblib')
joblib.dump(preprocessor, '../models/Productivity/preprocessor.joblib')

['../models/Productivity/preprocessor.joblib']